In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_regression 

In [2]:
df = pd.read_csv('data/ai_job_market.csv')
print(df.head())


   job_id                  job_title company_size company_industry    country  \
0       1                AI Engineer      Startup           Retail     Canada   
1       2  Machine Learning Engineer          MNC       Technology  Australia   
2       3  Machine Learning Engineer          MNC       Technology    Germany   
3       4           Business Analyst      Startup       Healthcare    Germany   
4       5             Data Scientist          MNC       Healthcare    Germany   

  remote_type experience_level  years_experience education_level  \
0      Remote           Senior                 2          Master   
1      Hybrid              Mid                 0        Bachelor   
2      Onsite              Mid                14          Master   
3      Remote              Mid                 9          Master   
4      Hybrid              Mid                 5          Master   

   skills_python  skills_sql  skills_ml  skills_deep_learning  skills_cloud  \
0              0         

**Classifying into categorical and numerical features**

In [3]:
categorical_features = [feature for feature in df.columns if df[feature].dtype!=int]
numerical_features = [feature for feature in df.columns if df[feature].dtype==int]  
print(f'The categorical features are: {categorical_features}')
print(f'The numerical features are: {numerical_features}')

The categorical features are: ['job_title', 'company_size', 'company_industry', 'country', 'remote_type', 'experience_level', 'education_level', 'hiring_urgency']
The numerical features are: ['job_id', 'years_experience', 'skills_python', 'skills_sql', 'skills_ml', 'skills_deep_learning', 'skills_cloud', 'salary', 'job_posting_month', 'job_posting_year', 'job_openings']


**Checking the correlation of numeric features with the target variable salary**

In [15]:
corr = df[numerical_features].corr()['salary'].drop('salary').sort_values(ascending=False)  #as salary is the target variable we are dropping it while checking the correlation
corr

skills_deep_learning    0.242504
skills_ml               0.232621
skills_cloud            0.153725
skills_python           0.009536
job_openings            0.005715
job_id                  0.003752
job_posting_year        0.003028
skills_sql             -0.003473
years_experience       -0.013266
job_posting_month      -0.013555
Name: salary, dtype: float64

**Finding the strong numeric features**

In [5]:
strong_numeric_features = corr[corr>0.01].index.to_list()
strong_numeric_features

['skills_deep_learning', 'skills_ml', 'skills_cloud']

**Classifying into nominal and ordinal features**

In [16]:
nominal_categorical_features = ['job_title','company_industry','country','remote_type','company_size']  #has no order within them
ordinal_categorical_features = ['experience_level','education_level','hiring_urgency']  #has orders within them


**Label encoding for nominal categorical features**

In [17]:
X_nominal_encoded = df[nominal_categorical_features].apply(LabelEncoder().fit_transform) 
X_nominal_encoded


,job_title,company_industry,country,remote_type,company_size
0,0,4,1,2,3
1,5,5,0,0,1
2,5,5,2,1,1
3,1,3,2,2,3
4,4,3,2,0,1
...,...,...,...,...,...
10340,3,5,6,1,3
10341,2,3,0,0,2
10342,5,1,2,0,3
10343,3,0,1,0,1


In [8]:
for feature in ordinal_categorical_features:
    print(f'{feature} :   {df[feature].unique()}')



experience_level :   <StringArray>
['Senior', 'Mid', 'Entry']
Length: 3, dtype: str
education_level :   <StringArray>
['Master', 'Bachelor', 'PhD']
Length: 3, dtype: str
hiring_urgency :   <StringArray>
['Low', 'High', 'Medium']
Length: 3, dtype: str


**Manual encoding for ordinal categorical features based on the orders of the features**

In [9]:
experience_level_map = {
    'Entry': 0,      # logical order: Entry → Mid → Senior
    'Mid': 1,
    'Senior': 2
}

education_level_map = {
    'Bachelor': 0,   # logical order: Bachelor → Master → PhD
    'Master': 1,
    'PhD': 2
}

hiring_urgency_map = {
    'Low': 0,        # logical order: Low → Medium → High
    'Medium': 1,
    'High': 2
}
X_ordinal_encoded = df[ordinal_categorical_features].copy()
X_ordinal_encoded['experience_level']  = df['experience_level'].map(experience_level_map)
X_ordinal_encoded['education_level']   = df['education_level'].map(education_level_map)
X_ordinal_encoded['hiring_urgency']    = df['hiring_urgency'].map(hiring_urgency_map)
X_ordinal_encoded

,experience_level,education_level,hiring_urgency
0,2,1,0
1,1,0,2
2,1,1,2
3,1,1,2
4,1,1,0
...,...,...,...
10340,0,2,0
10341,0,1,1
10342,0,2,0
10343,1,2,2


**Final encoded categorical features**

In [13]:
X_encoded_categorical_features = pd.concat([X_nominal_encoded,X_ordinal_encoded],axis=1)
X_encoded_categorical_features

,job_title,company_industry,country,remote_type,company_size,experience_level,education_level,hiring_urgency
0,0,4,1,2,3,2,1,0
1,5,5,0,0,1,1,0,2
2,5,5,2,1,1,1,1,2
3,1,3,2,2,3,1,1,2
4,4,3,2,0,1,1,1,0
...,...,...,...,...,...,...,...,...
10340,3,5,6,1,3,0,2,0
10341,2,3,0,0,2,0,1,1
10342,5,1,2,0,3,0,2,0
10343,3,0,1,0,1,1,2,2


**Statistical Ktest to select best categorical features based on pvalue or F statistics** 

In [11]:
target = df['salary']
selector = SelectKBest(score_func=f_regression, k='all')  #since we are predicting not classifying, so we are using f_regression
selector.fit(X_encoded_categorical_features, target)


,"score_func score_func: callable, default=f_classifFunction taking two arrays X and y, and returning a pair of arrays(scores, pvalues) or a single array with scores.Default is f_classif (see below ""See Also""). The default function onlyworks with classification tasks... versionadded:: 0.18",<function f_r...001E2CB96F950>
,"k k: int or ""all"", default=10Number of top features to select.The ""all"" option bypasses selection, for use in a parameter search.",'all'


**Categorical features that have strong relationship with the target variable**

In [18]:
strong_categorical_features = X_encoded_categorical_features.columns[selector.pvalues_<0.05].tolist()  #p_value lesser than 0.05 means f statistics higher
print(f'The categorical featurees that has strong relation with target variable salary is: {strong_categorical_features}')

The categorical featurees that has strong relation with target variable salary is: ['company_size', 'experience_level', 'hiring_urgency']


**Final important features containing both numeric and categorical features**

In [14]:
important_features = strong_numeric_features + strong_categorical_features
important_features

['skills_deep_learning',
 'skills_ml',
 'skills_cloud',
 'company_size',
 'experience_level',
 'hiring_urgency']